# Exact-money columnar exports

Offline synthetic rows, not market observations. Install `openport[pandas,arrow]`. Money stays decimal text; use Decimal for calculations. Nested objects export as JSON text. No data is downloaded or traded.

In [ ]:
from decimal import Decimal
from pathlib import Path
from tempfile import TemporaryDirectory
import pyarrow.parquet as pq
from openport.frames import fills_frame, arrow_table, write_parquet, write_arrow

fills = {"fills": [{"id": "1", "price": "123.450000", "fee": "0.65", "time": None, "context": {"source": "synthetic notebook"}}]}
fills_frame(fills)

In [ ]:
table = arrow_table(fills, "fills")
assert table["price"][0].as_py() == "123.450000"
assert Decimal(table["price"][0].as_py()) == Decimal("123.45")
with TemporaryDirectory() as directory:
    parquet = Path(directory) / "fills.parquet"
    write_parquet(fills, parquet, "fills")
    write_arrow(fills, Path(directory) / "fills.arrow", "fills")
    assert pq.read_table(parquet).equals(table)
table.schema

For your own server, pass `client.fills()`, `client.trades()`, `client.equity()`, `client.chain("SPX")`, or `client.get_backtest(id)` with kind `fills`, `trades`, `equity`, `chain`, or `backtest`. Backtest exports accept `section="days"` or `"attempts"`; select a sweep variant’s report first. Missing timestamps remain null. Analytics quote numbers remain numeric.